# Step 4: Generate Co-registration Table
Combines seed landmarks (from step_2) + auto-matched pairs (from step_3) into a final
one-to-one co-registration table.

Output: `{subject}_{date}_coreg_table.csv` → `/scratch/{subject}_{date}_coreg_cpsam/`

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path('.')))

from coreg_data_loading import (
    find_coreg_dirs, parse_coreg_dir_name, load_landmarks
)

%load_ext autoreload
%autoreload 2

DATA_DIR = Path('/root/capsule/data')
SCRATCH_DIR = Path('/root/capsule/scratch')

## Parameters

In [ ]:
# ── User parameters ───────────────────────────────────────────────────────────
subject_id = '790322'
czstack_date = '2025-07-10'

## Load Matching Results

In [ ]:
save_dir = SCRATCH_DIR / f'{subject_id}_{czstack_date}_coreg_cpsam'

# Path to auto-matching outputs (from step_3)
final_lm_path  = save_dir / f'{subject_id}_{czstack_date}_landmarks_auto_final.csv'
match_meta_path = save_dir / f'{subject_id}_{czstack_date}_auto_match_metadata.csv'

# Fall back to manual landmarks if auto not available
if final_lm_path.exists():
    final_landmarks = load_landmarks(final_lm_path)
    print(f'Loaded auto landmarks: {len(final_landmarks)} rows, '
          f'{final_landmarks["active"].sum()} active')
else:
    # Try manual landmarks from coreg dir
    coreg_dirs = find_coreg_dirs(DATA_DIR)
    coreg_dir = next(
        (d for d in coreg_dirs if parse_coreg_dir_name(d) == (subject_id, czstack_date)),
        None,
    )
    if coreg_dir is None:
        raise FileNotFoundError(f'No coreg dir for {subject_id}; run step_2 and step_3 first.')
    lm_files = sorted(coreg_dir.glob('*landmarks*qced*.csv'))
    if not lm_files:
        lm_files = sorted(coreg_dir.glob('*landmarks*.csv'))
    if not lm_files:
        raise FileNotFoundError(f'No landmarks in {coreg_dir}')
    final_landmarks = load_landmarks(lm_files[-1])
    print(f'Loaded manual landmarks: {lm_files[-1].name}, '
          f'{len(final_landmarks)} rows, {final_landmarks["active"].sum()} active')

# Load match metadata (iter_matched, match_probability)
if match_meta_path.exists():
    match_meta = pd.read_csv(match_meta_path)
    print(f'Match metadata: {len(match_meta)} auto-matched pairs')
else:
    match_meta = pd.DataFrame(columns=['czstack_cell_id', 'hcr_cell_id',
                                        'iter_matched', 'match_probability'])
    print('No match metadata — treating all as seed matches (iter=0)')

## Parse Landmark IDs → Coreg Table

In [ ]:
def parse_landmark_ids(lm_df: pd.DataFrame) -> pd.DataFrame:
    """Extract czstack_cell_id and hcr_cell_id from landmark ids column.
    Handles both 'cz{N}-hcr{M}' and 'qced_cz{N}-hcr{M}' formats.
    """
    rows = []
    for _, r in lm_df[lm_df['active'] == True].iterrows():
        s = str(r['ids'])
        # Strip qced_ prefix if present
        s = s.replace('qced_', '')
        if 'cz' in s and '-hcr' in s:
            try:
                cz_id  = int(s.split('-hcr')[0].replace('cz', ''))
                hcr_id = int(s.split('-hcr')[1])
                rows.append({'czstack_cell_id': cz_id, 'hcr_cell_id': hcr_id})
            except Exception:
                pass
    return pd.DataFrame(rows)


coreg_pairs = parse_landmark_ids(final_landmarks)
print(f'Parsed {len(coreg_pairs)} matched pairs from landmarks')

# Verify one-to-one
dup_cz  = coreg_pairs['czstack_cell_id'].duplicated().sum()
dup_hcr = coreg_pairs['hcr_cell_id'].duplicated().sum()
if dup_cz or dup_hcr:
    print(f'WARNING: {dup_cz} duplicate czstack_ids, {dup_hcr} duplicate hcr_ids — keeping first')
    coreg_pairs = coreg_pairs.drop_duplicates('czstack_cell_id').drop_duplicates('hcr_cell_id')
else:
    print('One-to-one: no duplicates.')

In [ ]:
# Merge in auto-match metadata (iter_matched, match_probability)
if len(match_meta) > 0:
    coreg_table = coreg_pairs.merge(
        match_meta[['czstack_cell_id', 'hcr_cell_id', 'iter_matched', 'match_probability']],
        on=['czstack_cell_id', 'hcr_cell_id'],
        how='left',
    )
    # Seeds have iter_matched=NaN → label as 0
    coreg_table['iter_matched'] = coreg_table['iter_matched'].fillna(0).astype(int)
else:
    coreg_table = coreg_pairs.copy()
    coreg_table['iter_matched'] = 0
    coreg_table['match_probability'] = np.nan

print(f'Coreg table shape: {coreg_table.shape}')
print('By iteration:')
print(coreg_table.groupby('iter_matched').size().to_string())
print('\nPreview:')
print(coreg_table.head())

## Save Coreg Table

In [ ]:
save_dir.mkdir(parents=True, exist_ok=True)
out_path = save_dir / f'{subject_id}_{czstack_date}_coreg_table.csv'
coreg_table.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'\nTotal matched: {len(coreg_table)}')
if 'match_probability' in coreg_table.columns and not coreg_table['match_probability'].isna().all():
    p = coreg_table['match_probability'].dropna()
    print(f'Auto-match probability: mean={p.mean():.3f}, min={p.min():.3f}')

## Comparison with Manual (if available)

In [ ]:
# Compare auto coreg table with manual ground truth (if available)
coreg_dirs = find_coreg_dirs(DATA_DIR)
coreg_dir = next(
    (d for d in coreg_dirs if parse_coreg_dir_name(d) == (subject_id, czstack_date)),
    None,
)
if coreg_dir is not None:
    manual_csvs = sorted(coreg_dir.glob(f'*coreg_table*.csv'))
    if manual_csvs:
        manual_table = pd.read_csv(manual_csvs[-1])
        # Normalise column names
        col_map = {c: c.replace('_id', '_cell_id') if '_id' in c and 'cell' not in c else c
                   for c in manual_table.columns}
        manual_table = manual_table.rename(columns=col_map)
        if 'czstack_cell_id' not in manual_table.columns:
            manual_table.columns = ['czstack_cell_id', 'hcr_cell_id']

        auto_set   = set(zip(coreg_table['czstack_cell_id'], coreg_table['hcr_cell_id']))
        manual_set = set(zip(manual_table['czstack_cell_id'], manual_table['hcr_cell_id']))

        tp = len(auto_set & manual_set)
        fp = len(auto_set - manual_set)  # auto but not manual
        fn = len(manual_set - auto_set)  # manual but not auto

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        print(f'=== vs Manual ({manual_csvs[-1].name}) ===')
        print(f'Manual pairs:   {len(manual_set)}')
        print(f'Auto pairs:     {len(auto_set)}')
        print(f'True positives: {tp}')
        print(f'False positives (auto, not manual): {fp}')
        print(f'False negatives (manual, not auto): {fn}')
        print(f'Precision: {precision:.3f}   Recall: {recall:.3f}   F1: {f1:.3f}')
    else:
        print('No manual coreg table found for comparison.')
else:
    print('Coreg dir not found — skipping comparison.')